# Door 1: from precomputed score events to a compiled or a fitted rule

This notebook accompanies the docs page [`door1-score-events`](../../docs/examples/door1-score-events.md). Each event is a Gaussian-location measurement $x\sim\mathcal N(\mu,1)$; at the reference point $\mu_0=0$ the score is $s(x)=x$. The notebook runs both public tasks on the same precomputed `ScoreSample`: `optimize_partition` with a stability certificate, the compile bridge into a reusable rule, and `fit_quantizer` as the direct alternative, at a larger, more decisive sample size than the docs page's fast snippets use.

## Data

`examples.synthetic_problems.gaussian_location` returns deterministic train/validation/test splits. Sample sizes shrink under `SCOREQUANT_EXAMPLE_FAST` through `example_scale`, so this notebook runs quickly in CI and at full scale locally.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.synthetic_problems import gaussian_location

sizes = (example_scale(4_000, 500), example_scale(1_000, 200), example_scale(10_000, 1_000))
problem = gaussian_location(sizes=sizes)
train, test = problem.train, problem.test
provenance = sq.ScoreProvenance(kind="exact", reference_point=(0.0,))
sample = sq.ScoreSample(train.scores, train.weights, provenance=provenance)
sample.scores.shape

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.asarray(train.observations[:, 0]), bins=40, density=True, alpha=0.75)
ax.set(xlabel="measurement x = score s(x)", ylabel="density", title="Training events");

## Partition, then compile

`optimize_partition` assigns labels to exactly this sample. `exchange_stable` and `best_remaining_gain` are the resulting certificate: whether any single relocation could still raise the exact objective.

In [ ]:
partition = sq.optimize_partition(
    sample.scores,
    weights=sample.weights,
    n_bins=4,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=12, initializer_restarts=8),
    provenance=sample.provenance,
)
stability = sq.exchange_stability_report(
    partition.training_scores, partition.labels, weights=sample.weights
)
partition.exchange_stable, stability.stable, float(partition.best_remaining_gain)

In [ ]:
compiled = partition.compile_quantizer()
compiled_report = compiled.evaluate_scores(test.scores, test.weights)
train_efficiency = float(partition.train_report.geometric_mean_retention)
compiled_efficiency = float(compiled_report.geometric_mean_retention)
train_efficiency, compiled_efficiency

## Fit directly, and compare

A direct `fit_quantizer` call with `NormalizedTrace` and k-means never goes through a partition at all. It reaches criteria and solvers the compile bridge cannot: only an exchange-stable, nonsingular `DOptimality` partition can be compiled.

In [ ]:
direct = sq.fit_quantizer(
    sample,
    n_bins=4,
    criterion=sq.NormalizedTrace(),
    config=sq.KMeansConfig(seed=12, solver_restarts=8),
)
direct_report = direct.evaluate_scores(test.scores, test.weights)
{
    "compiled test D-efficiency": float(compiled_report.geometric_mean_retention),
    "direct-fit test D-efficiency": float(direct_report.geometric_mean_retention),
}

## When to use which

On this one-dimensional problem the compiled D-optimal rule and the directly fitted normalized-trace rule land on almost the same held-out D-efficiency: both are approximating the same optimal interval partition. That agreement is a property of this simple problem, not a guarantee. Use the compile bridge when the D-optimal exchange result and its exchange-stability certificate are what you wanted anyway; use `fit_quantizer` directly when you know from the start you want a rule, or need `ProfiledDOptimality`, soft Voronoi, or the scalar dynamic program — none of which the compile bridge reaches.